<a href="https://colab.research.google.com/github/amilafr/algo-python-pro2/blob/main/M5L4_The_main_game_mode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M5L4 The main game mode

[PPT M5L4 ENG](https://docs.google.com/presentation/d/1YERkAfoYtaNJy1WvT5sfsfFzCglixwHa73fzZdK1kbI/edit?usp=sharing)

# game.py

In [ ]:
# import library
from direct.showbase.ShowBase import ShowBase
# mapmanager
from mapmanager import Mapmanager
# hero
from hero import Hero

class Game(ShowBase):
    # constructor
    def __init__(self):
        ShowBase.__init__(self)
        # map
        self.land = Mapmanager()
        self.land.loadLand('land.txt') # load file txt map
        # self.land.loadLand('land2.txt')
        self.hero = Hero((11, 10, 2), self.land) # player
        base.camLens.setFov(90)

# bikin scene
game = Game()

# run
game.run()

# mapmanager.py

In [ ]:
class Mapmanager():
    # constructor
    def __init__(self):
        self.model = 'block.egg'
        self.texture = 'block.png'
        # self.color = (0.2, 0.2, 0.35, 1) # rgba
        self.colors = [
            (0.5, 0.3, 0.0, 1),
            (0.2, 0.2, 0.3, 1),
            (0.5, 0.5, 0.2, 1),
            (0.0, 0.6, 0.0, 1)
        ]

        self.startNew()
        # self.addBlock((0, 10, 0))

    # pasang object
    def startNew(self):
        self.land = render.attachNewNode('Land') # bikin node baru

    # nambah block
    def addBlock(self, position):
        self.block = loader.loadModel(self.model)
        self.block.setTexture(loader.loadTexture(self.texture))
        self.block.setPos(position)

        self.color = self.getColor(position[2]) # get colornya
        self.block.setColor(self.color)

        self.block.reparentTo(self.land)

        '''add this'''
        self.block.setTag("at", str(position))

    def getColor(self, z):
        if z < len(self.colors):
            return self.colors[z]
        else:
            return self.colors[len(self.colors)-1] # color terakhir

    # hapus map sebelumnya
    def clear(self):
        self.land.removeNode()
        self.startNew()

    # load file map
    def loadLand(self, filename):
        self.clear() # clear dulu map sebelumnya
        with open(filename) as file:
            y = 0
            for line in file:
                x = 0
                line = line.split(' ')
                for z in line:
                    for z0 in range(int(z) + 1):
                        block = self.addBlock((x, y, z0))
                    x += 1
                y += 1

    '''ADD THIS'''
    # ngecek depannya kosong atau nggak
    def isEmpty(self, pos):
        blocks = self.findBlocks(pos)
        if blocks:
            return False
        else:
            return True

    # ngecek koordinat block nya
    def findBlocks(self, pos):
        return self.land.findAllMatches("=at=" + str(pos))

    # ngecek ketinggian/jumlah block
    def findHighestEmpty(self, pos):
        x, y, z = pos
        z = 1
        while not self.isEmpty((x, y, z)):
            z += 1
        return (x, y, z)

    def delBlock(self, pos):
        blocks = self.findBlocks(pos)
        for block in blocks:
            block.removeNode()

    def buildBlock(self, pos):
        x, y, z = pos
        new = self.findHighestEmpty(pos)
        if new[2] <= z+1:
            self.addBlock(new)

    def delBlockFrom(self, pos):
        x, y, z = pos
        p = x, y, z-1
        blocks = self.findBlocks(p)
        for block in blocks:
            block.removeNode()



# hero.py

In [ ]:
class Hero():
    # constructor
    def __init__(self, pos, land):
        self.land = land # map
        self.hero = loader.loadModel('smiley') # load model
        self.hero.setColor(1, 0.5, 0) # warna
        self.hero.setScale(0.3) # ukuran
        self.hero.setPos(pos) # posisi
        self.hero.reparentTo(render)

        self.cameraBind()
        self.accept_events()

        self.mode = True

    # camera --> player mode
    def cameraBind(self):
        base.disableMouse() # off mouse
        base.camera.setH(180) # rotate kamera biar ngelihat ke depan player
        base.camera.reparentTo(self.hero) # camera-nya sesuai player
        base.camera.setPos(0, 0, 1.5)
        self.cameraOn = True

    # camera ngelihat dari atas
    def cameraUp(self):
        pos = self.hero.getPos() #posisi player

        base.mouseInterfaceNode.setPos(-pos[0], -pos[1], -pos[2]-3)
        base.camera.reparentTo(render)
        base.enableMouse()
        self.cameraOn = False

    def changeView(self):
        if self.cameraOn:
            self.cameraUp()
        else:
            self.cameraBind()

    # noleh kiri
    def turn_left(self):
        self.hero.setH((self.hero.getH() + 5) % 360)

    # noleh kanan
    def turn_right(self):
        self.hero.setH((self.hero.getH() - 5) % 360)

    def look_at(self, angle): # lagi lihat mana camera player
        # posisi player sekarang
        from_x = round(self.hero.getX())
        from_y = round(self.hero.getY())
        from_z = round(self.hero.getZ())

        dx, dy = self.check_dir(angle)

        return from_x + dx, from_y + dy, from_z

    def check_dir(self, angle): # cek arah gerak
        if angle >= 0 and angle <= 20:
            return 0, -1
        elif angle >= 20 and angle <= 65:
            return 1, -1
        elif angle > 65 and angle <= 110:
            return 1, 0
        elif angle > 110 and angle <= 155:
            return 1, 1
        elif angle > 155 and angle <= 200:
            return 0, 1
        elif angle > 200 and angle <= 245:
            return -1, 1
        elif angle > 245 and angle <= 290:
            return -1, 0
        elif angle > 290 and angle <= 335:
            return -1, -1
        else:
            return 0, -1

    def just_move(self, angle):
        pos = self.look_at(angle)
        self.hero.setPos(pos)

    '''ADD THIS'''
    def try_move(self, angle):
        pos = self.look_at(angle)

        if self.land.isEmpty(pos): # depannya kosong
            pos = self.land.findHighestEmpty(pos)
            self.hero.setPos(pos)
        else:
            pos = pos[0], pos[1], pos[2] + 1
            if self.land.isEmpty(pos):
                self.hero.setPos(pos)

    def move_to(self, angle):
        if self.mode == True:
            self.just_move(angle)
        else:
            self.try_move(angle)

    def back(self):
        angle = (self.hero.getH() + 180) % 360
        self.move_to(angle)

    def forward(self):
        angle = (self.hero.getH()) % 360
        self.move_to(angle)

    def left(self):
        angle = (self.hero.getH() + 90) % 360
        self.move_to(angle)

    def right(self):
        angle = (self.hero.getH() + 270) % 360
        self.move_to(angle)

    '''ADD THIS'''
    # naik ke atas
    def up(self):
        self.hero.setZ(self.hero.getZ() + 1)

    # ke bawah
    def down(self):
        self.hero.setZ(self.hero.getZ() - 1)

    # ganti mode
    def changeMode(self):
        if self.mode == True:
            self.mode = False
        else:
            self.mode = True

    # nambahin block di depannya
    def build(self):
        angle = self.hero.getH() % 360
        pos = self.look_at(angle)
        if self.mode:
            self.land.addBlock(pos)
        else:
            self.land.buildBlock(pos)

    # hapus block
    def destroy(self):
        angle = self.hero.getH() % 360
        pos = self.look_at(angle)
        if self.mode:
            self.land.delBlock(pos)
        else:
            self.land.delBlockFrom(pos)

    # events
    def accept_events(self):
        base.accept('c', self.changeView)

        base.accept('n', self.turn_left)
        base.accept('n'+'-repeat', self.turn_left)

        base.accept('m', self.turn_right)
        base.accept('m'+'-repeat', self.turn_right)

        base.accept('m', self.turn_right)
        base.accept('m'+'-repeat', self.turn_right)

        # gerak
        base.accept('w', self.forward)
        base.accept('w'+'-repeat', self.forward)

        base.accept('s', self.back)
        base.accept('s'+'-repeat', self.back)

        base.accept('a', self.left)
        base.accept('a'+'-repeat', self.left)

        base.accept('d', self.right)
        base.accept('d'+'-repeat', self.right)

        # atas-bawah
        base.accept('e', self.up)
        base.accept('e'+'-repeat', self.up)

        base.accept('q', self.down)
        base.accept('q'+'-repeat', self.down)

        # ganti mode
        base.accept('z', self.changeMode)

        # nambah dan hapus block
        base.accept('b', self.build)
        base.accept('v', self.destroy)